# CIC-IDS-2017 data exploration

We examine the merged CIC-IDS-2017 flow records to decide which records and measurements to keep. The [dataset reference](../DATASET.md) explains the column names, units and flow directions.

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def show_table(frame, row_label=None):
    """Display a table with one header row and no extra index."""
    if row_label is not None:
        frame = frame.rename_axis(row_label).reset_index()
    display({"text/html": frame.to_html(index=False),
             "text/plain": frame.to_string(index=False)}, raw=True)

## 2. Load the dataset

In [2]:
dataset_candidates = [
    Path("../data/raw/cicids2017_merged.csv"),
    Path("ml/data/raw/cicids2017_merged.csv"),
]

DATASET_PATH = next(
    (path.resolve() for path in dataset_candidates if path.exists()),
    None,
)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not find ml/data/raw/cicids2017_merged.csv. "
        "Run the notebook from the repository root or ml/notebooks."
    )

df = pd.read_csv(DATASET_PATH, low_memory=False)
print("Dataset loaded successfully.")

Dataset loaded successfully.


## 3. Dataset overview

In this section, we examine the dimensions of the dataset, the columns that describe each flow, and the types of values recorded in them.

### 3.1 Dimensions and column names

In [3]:
row_count, column_count = df.shape

print(f"Rows: {row_count:,}")
print(f"Columns: {column_count}")
print("\nColumn names:")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

duplicate_columns = df.columns[df.columns.duplicated()].tolist()
if duplicate_columns:
    raise ValueError(f"Repeated names in the loaded dataframe: {duplicate_columns}")

Rows: 3,119,345
Columns: 84

Column names:
1. Flow ID
2. Src IP
3. Src Port
4. Dst IP
5. Dst Port
6. Protocol
7. Timestamp
8. Flow Duration
9. Total Fwd Packet
10. Total Bwd packets
11. Total Length of Fwd Packet
12. Total Length of Bwd Packet
13. Fwd Packet Length Max
14. Fwd Packet Length Min
15. Fwd Packet Length Mean
16. Fwd Packet Length Std
17. Bwd Packet Length Max
18. Bwd Packet Length Min
19. Bwd Packet Length Mean
20. Bwd Packet Length Std
21. Flow Bytes/s
22. Flow Packets/s
23. Flow IAT Mean
24. Flow IAT Std
25. Flow IAT Max
26. Flow IAT Min
27. Fwd IAT Total
28. Fwd IAT Mean
29. Fwd IAT Std
30. Fwd IAT Max
31. Fwd IAT Min
32. Bwd IAT Total
33. Bwd IAT Mean
34. Bwd IAT Std
35. Bwd IAT Max
36. Bwd IAT Min
37. Fwd PSH Flags
38. Bwd PSH Flags
39. Fwd URG Flags
40. Bwd URG Flags
41. Fwd Header Length
42. Bwd Header Length
43. Fwd Packets/s
44. Bwd Packets/s
45. Packet Length Min
46. Packet Length Max
47. Packet Length Mean
48. Packet Length Std
49. Packet Length Variance
50. FIN

The loaded dataframe has 84 distinct column names.

### 3.2 Sample records and data types

In [4]:
display(df.head())

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443.0,192.168.10.5,54865.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80.0,192.168.10.5,55054.0,6.0,7/7/2017 3:30,109.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80.0,192.168.10.5,55055.0,6.0,7/7/2017 3:30,52.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443.0,192.168.10.16,46236.0,6.0,7/7/2017 3:30,34.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443.0,192.168.10.5,54863.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [5]:
show_table(df.dtypes.astype(str).value_counts().to_frame("Columns"), "Loaded dtype")
print(f"Dataframe memory: {df.memory_usage(index=True, deep=True).sum() / 1024**3:.2f} GiB")

Loaded dtype,Columns
float64,79
object,5


Dataframe memory: 2.85 GiB


The five text columns (`object`) are `Flow ID`, `Src IP`, `Dst IP`, `Timestamp`, and `Label`. The remaining 79 are stored as floating-point numbers. A mean packet length can be fractional, but ports, protocol codes, packet and flag counts, byte counts and lengths, TCP window sizes, and times recorded in whole microseconds should be whole numbers.

### 3.3 Whole-number measurements

Here, we test whether any finite value in these 44 fields has a fractional part. A port recorded as `443.0` represents the integer 443; `443.5`, by contrast, cannot represent a port.

In [6]:
integer_fields = [
    "Src Port", "Dst Port", "Protocol", "Flow Duration", "Total Fwd Packet", "Total Bwd packets",
    "Total Length of Fwd Packet", "Total Length of Bwd Packet", "Fwd Act Data Pkts",
    "FWD Init Win Bytes", "Bwd Init Win Bytes", "Fwd Header Length", "Bwd Header Length",
    "Fwd Seg Size Min", "Fwd IAT Total", "Bwd IAT Total",
    *(f"{stem} {stat}" for stem in ("Fwd Packet Length", "Bwd Packet Length", "Packet Length",
                                    "Flow IAT", "Fwd IAT", "Bwd IAT", "Active", "Idle")
      for stat in ("Min", "Max")),
    *(f"{side} {flag} Flags" for side in ("Fwd", "Bwd") for flag in ("PSH", "URG")),
    *(f"{flag} Flag Count" for flag in ("FIN", "SYN", "RST", "PSH", "ACK", "URG", "CWR", "ECE")),
]

def count_fractional(values):
    finite = values[np.isfinite(values)]
    return int(np.count_nonzero(finite != np.trunc(finite)))

fractional = pd.Series({c: count_fractional(df[c].to_numpy()) for c in integer_fields})
print(f"Whole-number fields checked: {len(fractional)}")
print(f"Finite values with a fractional part: {fractional.sum():,}")
if fractional.any():
    show_table(fractional[fractional.gt(0)].to_frame("Fractional values"), "Feature")
    raise ValueError("Fractional values found in fields defined as whole numbers.")

Whole-number fields checked: 44
Finite values with a fractional part: 0


The values in these fields are integral despite their floating-point representation, so we retain them as recorded. Integrality alone does not establish validity: a negative packet count, for example, would also be a whole number.

## 4. Target labels

In this section, we examine which rows have a target label, how the labeled flows are distributed across classes, and how the class names are recorded.

### 4.1 Records without a label

A flow without a known class cannot serve as a labeled example. We therefore exclude rows where `Label` is missing.

In [7]:
invalid_label_mask = df["Label"].isna()
display(df.loc[invalid_label_mask].head())

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
1692131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692134,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
rows_before = len(df)
df.dropna(subset=["Label"], inplace=True)
df.reset_index(drop=True, inplace=True)
df_valid = df
del df
del invalid_label_mask

population = pd.DataFrame({
    "Population": ["Original CSV", "Unlabeled rows removed", "Labeled rows retained"],
    "Rows": [rows_before, rows_before - len(df_valid), len(df_valid)],
})
population["% of original rows"] = (100 * population["Rows"] / rows_before).round(2)
show_table(population)

Population,Rows,% of original rows
Original CSV,3119345,100.00
Unlabeled rows removed,288602,9.25
Labeled rows retained,2830743,90.75


### 4.2 Label distribution

In [9]:
label_counts = df_valid["Label"].value_counts(dropna=False)
label_distribution = pd.DataFrame(
    {
        "count": label_counts,
        "percentage": (label_counts / len(df_valid) * 100).round(6),
    }
)

print(f"Unique detailed labels: {df_valid['Label'].nunique(dropna=False)}")
show_table(label_distribution, "Label")

Unique detailed labels: 15


Label,count,percentage
BENIGN,2273097,80.300366
DoS Hulk,231073,8.162981
PortScan,158930,5.614427
DDoS,128027,4.522735
DoS GoldenEye,10293,0.363615
FTP-Patator,7938,0.280421
SSH-Patator,5897,0.208320
DoS slowloris,5796,0.204752
DoS Slowhttptest,5499,0.194260
Bot,1966,0.069452


BENIGN accounts for 80.30% of the labeled flows, whereas Heartbleed has only 11 records. A loss of even a few flows can therefore represent a substantial share of a rare class. The three Web Attack names also share a malformed separator between “Web Attack” and the attack name.

### 4.3 Web Attack label separator

We replace only the malformed character after `Web Attack` with a hyphen. The attack name following the separator is preserved.

In [10]:
web_labels = df_valid["Label"].str.startswith("Web Attack", na=False)
counts = df_valid.loc[web_labels, "Label"].value_counts()
# U+0096 is the separator in this CSV, not part of an attack name.
normalized = counts.index.str.replace(r"^Web Attack \x96 ", "Web Attack - ", regex=True)
expected = {"Web Attack - Brute Force", "Web Attack - XSS", "Web Attack - Sql Injection"}
observed = set(normalized)
if observed != expected:
    raise ValueError(f"Web Attack labels: missing={sorted(expected - observed)}, "
                     f"unexpected={sorted(observed - expected)}")

changes = pd.DataFrame({
    "Original label": counts.index,
    "Normalized label": normalized,
    "Rows affected": counts.to_numpy(),
})
changes = changes.loc[changes["Original label"] != changes["Normalized label"]]
replacements = dict(zip(changes["Original label"], changes["Normalized label"]))
df_valid["Label"] = df_valid["Label"].replace(replacements)
show_table(changes)

Original label,Normalized label,Rows affected
Web Attack  Brute Force,Web Attack - Brute Force,1507
Web Attack  XSS,Web Attack - XSS,652
Web Attack  Sql Injection,Web Attack - Sql Injection,21


## 5. Missing and infinite measurements

A label identifies the class of a flow, but its measurements may still be incomplete. We distinguish missing values from positive or negative infinity when counting the affected features.

### 5.1 Affected features

In [11]:
feature_columns = df_valid.columns.drop("Label")
numeric_columns = df_valid.select_dtypes(include="number").columns
quality = pd.DataFrame({
    "NaN in the CSV": df_valid[feature_columns].isna().sum(),
    "Infinite in the CSV": pd.Series({
        name: int(np.isinf(df_valid[name]).sum()) for name in numeric_columns
    }).reindex(feature_columns, fill_value=0),
})
show_table(quality.loc[quality.any(axis=1)], "Feature")

Feature,NaN in the CSV,Infinite in the CSV
Flow Bytes/s,1358,1509
Flow Packets/s,0,2867


Only `Flow Bytes/s` and `Flow Packets/s` contain non-finite values. Both rates divide a traffic total by the elapsed time. A recorded duration of zero is therefore a possible explanation for both columns.

### 5.2 Rate availability by flow duration

In [12]:
rate_columns = ["Flow Bytes/s", "Flow Packets/s"]
zero_duration_mask = df_valid["Flow Duration"].eq(0)
nonfinite_rates = ~np.isfinite(df_valid[rate_columns])
comparison = pd.crosstab(zero_duration_mask, nonfinite_rates.sum(axis=1)).reindex(
    index=[True, False], columns=[0, 1, 2], fill_value=0
)
comparison.index = pd.Index(["Zero", "Nonzero"], name="Recorded duration")
comparison.columns = ["Both rates finite", "One rate non-finite", "Both rates non-finite"]
show_table(comparison, "Recorded duration")
if not all(nonfinite_rates[c].equals(zero_duration_mask) for c in rate_columns):
    raise ValueError("The two rate masks do not match zero duration; review before removing rows.")

# Use one representation for an undefined rate; do not substitute a numeric rate.
df_valid[rate_columns] = df_valid[rate_columns].replace([np.inf, -np.inf], np.nan)

Recorded duration,Both rates finite,One rate non-finite,Both rates non-finite
Zero,0,0,2867
Nonzero,2827876,0,0


Both rates are undefined on exactly the 2,867 flows with zero recorded duration. On every flow with nonzero duration, both are finite. This accounts for the unavailable rates without implying that the packet records themselves are invalid.

## 6. Zero-duration flows

Several packets can share the same recorded timestamp. A duration of zero therefore does not, by itself, tell us that the record is wrong.

### 6.1 Packet, payload, and timing evidence

If several packets share a recorded timestamp, their inter-arrival times (IATs) should all be zero. We examine packet counts and payload sizes for these flows, then compare their recorded IATs with that expectation.

In [13]:
zero_flows = df_valid.loc[zero_duration_mask]
packets = zero_flows["Total Fwd Packet"] + zero_flows["Total Bwd packets"]
payload = zero_flows["Total Length of Fwd Packet"] + zero_flows["Total Length of Bwd Packet"]
show_table(payload.groupby(packets.rename("Total packets")).agg(
    Flows="size", **{"Minimum payload (bytes)": "min", "Maximum payload (bytes)": "max"}
), "Total packets")

iat_features = numeric_columns[numeric_columns.str.contains("IAT")].tolist()
iat = zero_flows[iat_features]
timing_exception = (iat.isna() | iat.ne(0)).any(axis=1)
show_table(pd.Series({
    "All IAT fields present and zero": int((~timing_exception).sum()),
    "At least one missing or nonzero IAT": int(timing_exception.sum()),
}, name="Flows").to_frame(), "Zero-duration timing check")

Total packets,Flows,Minimum payload (bytes),Maximum payload (bytes)
2.0,2865,0.0,2065.0
3.0,2,0.0,18.0


Zero-duration timing check,Flows
All IAT fields present and zero,2866
At least one missing or nonzero IAT,1


Every zero-duration flow contains two or three packets. In 2,866 flows, all IATs are present and zero, consistent with packets sharing a timestamp. The table below isolates the one exception.

In [14]:
show_table(zero_flows.loc[timing_exception, [
    "Flow Duration", "Total Fwd Packet", "Total Bwd packets", "Flow IAT Min", "Fwd IAT Min",
]])

Flow Duration,Total Fwd Packet,Total Bwd packets,Flow IAT Min,Fwd IAT Min
0.0,2.0,1.0,-1.0,1.0


The exception has two forward packets and one backward packet. Its shortest recorded gap is **−1 µs**, while its two forward packets are **1 µs** apart despite a recorded duration of zero. The measurements disagree; they do not establish why.

### 6.2 Removal of flows with undefined rates

We keep both rate features. For flows with zero recorded duration, neither rate can be calculated, and substituting zero would suggest no traffic even when packets were recorded. We exclude these flows and report the percentage removed from each label.

In [15]:
def remove_and_report(frame, mask):
    """Remove flagged rows once. Percentages use each label's pre-removal support."""
    before = frame["Label"].value_counts()
    removed = frame.loc[mask, "Label"].value_counts()
    summary = pd.DataFrame({
        "Rows before removal": before.reindex(removed.index),
        "Rows removed": removed,
    })
    summary.loc["Total"] = [len(frame), int(mask.sum())]
    summary["% of label removed"] = 100 * summary["Rows removed"] / summary["Rows before removal"]
    summary["Rows remaining"] = summary["Rows before removal"] - summary["Rows removed"]

    frame.drop(index=frame.index[mask], inplace=True)
    frame.reset_index(drop=True, inplace=True)
    assert summary.loc["Total", "Rows remaining"] == len(frame)
    show_table(summary.round(4), "Label")


label_counts_before_cleaning = df_valid["Label"].value_counts()
remove_and_report(df_valid, zero_duration_mask)

missing_counts = df_valid.isna().sum()
if missing_counts.any():
    remaining = missing_counts[missing_counts.gt(0)].sort_values(ascending=False)
    show_table(remaining.to_frame("Missing cells"), "Feature")
    raise ValueError(f"{int(remaining.sum()):,} missing cells remain after zero-duration removal.")

Label,Rows before removal,Rows removed,% of label removed,Rows remaining
BENIGN,2273097,1777,0.0782,2271320
DoS Hulk,231073,949,0.4107,230124
PortScan,158930,126,0.0793,158804
Bot,1966,10,0.5086,1956
FTP-Patator,7938,3,0.0378,7935
DDoS,128027,2,0.0016,128025
Total,2830743,2867,0.1013,2827876


These exclusions affect about 0.10% of labeled flows and leave no missing cells. The flow with the negative IAT is included in this count.

## 7. Numeric ranges and negative measurements

Durations and byte counts cannot be negative. Two fields need different treatment: `FWD Init Win Bytes` and `Bwd Init Win Bytes` use −1 to mean that a TCP window is unavailable. We check these separately from the measurements that must be nonnegative.

### 7.1 Numeric feature ranges

In [16]:
numeric_summary = df_valid.loc[:, numeric_columns].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    show_table(numeric_summary, "Feature")

Feature,count,mean,std,min,1%,25%,50%,75%,99%,max
Src Port,2827876.0,4.113793e+04,2.229161e+04,0.000000e+00,80.000000,32784.000000,50949.000000,5.842000e+04,6.490900e+04,6.553500e+04
Dst Port,2827876.0,8.061534e+03,1.827432e+04,0.000000e+00,22.000000,53.000000,80.000000,4.430000e+02,6.165000e+04,6.553500e+04
Protocol,2827876.0,9.884209e+00,5.263117e+00,0.000000e+00,6.000000,6.000000,6.000000,1.700000e+01,1.700000e+01,1.700000e+01
Flow Duration,2827876.0,1.480065e+07,3.366750e+07,-1.300000e+01,1.000000,155.000000,31338.000000,3.239368e+06,1.177786e+08,1.200000e+08
Total Fwd Packet,2827876.0,9.368972e+00,7.500527e+02,1.000000e+00,1.000000,2.000000,2.000000,5.000000e+00,4.800000e+01,2.197590e+05
Total Bwd packets,2827876.0,1.040396e+01,9.978937e+02,0.000000e+00,0.000000,1.000000,2.000000,4.000000e+00,5.700000e+01,2.919220e+05
Total Length of Fwd Packet,2827876.0,5.498522e+02,9.998639e+03,0.000000e+00,0.000000,12.000000,62.000000,1.880000e+02,1.159500e+04,1.290000e+07
Total Length of Bwd Packet,2827876.0,1.617903e+04,2.264235e+06,0.000000e+00,0.000000,2.000000,123.000000,4.840000e+02,7.190800e+04,6.554530e+08
Fwd Packet Length Max,2827876.0,2.078044e+02,7.175183e+02,0.000000e+00,0.000000,6.000000,37.000000,8.100000e+01,2.940000e+03,2.482000e+04
Fwd Packet Length Min,2827876.0,1.872929e+01,6.035533e+01,0.000000e+00,0.000000,0.000000,2.000000,3.600000e+01,7.700000e+01,2.325000e+03


The minimum `Flow Duration` is −13 µs, the minimum `Flow IAT Min` is −14 µs, and some recorded header lengths are negative. These minima establish that invalid measurements occur, but not how many distinct flows they affect.

### 7.2 Affected features and window sentinels

The first table counts negative values in each affected feature, excluding the two window fields. The second distinguishes valid −1 window sentinels from values outside the allowed range. A flow can appear under several features, so the first table is not a removal count.

In [17]:
window_sentinel_features = ["FWD Init Win Bytes", "Bwd Init Win Bytes"]
negative_counts = pd.Series({
    c: int(df_valid[c].lt(0).sum()) for c in numeric_columns if c not in window_sentinel_features
})
show_table(negative_counts[negative_counts.gt(0)].sort_values(ascending=False)
        .to_frame("Rows with a negative value"), "Feature")

windows = df_valid[window_sentinel_features]
window_counts = pd.DataFrame({
    "Rows equal to −1": windows.eq(-1).sum(),
    "Rows outside the allowed values": (
        windows.lt(-1) | windows.gt(65535)
    ).sum(),
})
show_table(window_counts, "Window feature")
if window_counts["Rows outside the allowed values"].any():
    raise ValueError("Unexpected TCP-window values; review before removal.")

Feature,Rows with a negative value
Flow IAT Min,2890
Flow Duration,115
Flow IAT Mean,115
Flow Packets/s,115
Flow IAT Max,115
Flow Bytes/s,85
Fwd Header Length,35
Fwd Seg Size Min,35
Bwd Header Length,22
Fwd IAT Min,17


Window feature,Rows equal to −1,Rows outside the allowed values
FWD Init Win Bytes,1001172,0
Bwd Init Win Bytes,1439672,0


Negative values appear in duration, IATs, rates and header or segment fields. The window columns, in contrast, contain many −1 sentinels but no invalid values. Treating every negative cell alike would discard valid records.

### 7.3 Removal and class impact

The removal rule uses `Flow Duration`, all flow and directional IAT fields, both header lengths, both average segment sizes, and the minimum forward segment size. Where several fields flag the same flow, it is removed only once. The feature table above shows the specific columns with negative values.

In [18]:
header_segment_features = [
    "Fwd Header Length", "Bwd Header Length", "Fwd Segment Size Avg",
    "Bwd Segment Size Avg", "Fwd Seg Size Min",
]
negative_duration = df_valid["Flow Duration"].lt(0)
negative_iat = df_valid[iat_features].lt(0).any(axis=1)
negative_header = df_valid[header_segment_features].lt(0).any(axis=1)
invalid_negative_mask = negative_duration | negative_iat | negative_header
print(f"{negative_iat.sum():,} flows have negative IATs; "
      f"{(negative_duration & negative_iat).sum():,} of the {negative_duration.sum():,} "
      f"negative-duration flows are among them.")
print(f"{(negative_header & ~(negative_duration | negative_iat)).sum():,} additional flows "
      "have negative header or segment measurements.")
remove_and_report(df_valid, invalid_negative_mask)

2,890 flows have negative IATs; 115 of the 115 negative-duration flows are among them.
35 additional flows have negative header or segment measurements.


Label,Rows before removal,Rows removed,% of label removed,Rows remaining
BENIGN,2271320,2731,0.1202,2268589
DoS Hulk,230124,159,0.0691,229965
DDoS,128025,19,0.0148,128006
DoS GoldenEye,10293,5,0.0486,10288
Heartbleed,11,4,36.3636,7
FTP-Patator,7935,4,0.0504,7931
SSH-Patator,5897,2,0.0339,5895
Infiltration,36,1,2.7778,35
Total,2827876,2925,0.1034,2824951


Four of the eleven Heartbleed flows had invalid measurements. Although the same rule applies to all classes, those four exclusions account for 36.36% of the labeled Heartbleed records.

## 8. Constant columns

The numeric summary showed several columns with the same minimum and maximum. We check their values directly: a column that never changes cannot distinguish one flow from another in these data.

In [19]:
constant_values = {c: df_valid[c].iloc[0] for c in feature_columns
                   if df_valid[c].eq(df_valid[c].iloc[0]).all()}
show_table(pd.Series(constant_values, name="Constant value").to_frame(), "Feature")

Feature,Constant value
Bwd PSH Flags,0.0
Bwd URG Flags,0.0
Fwd Bytes/Bulk Avg,0.0
Fwd Packet/Bulk Avg,0.0
Fwd Bulk Rate Avg,0.0
Bwd Bytes/Bulk Avg,0.0
Bwd Packet/Bulk Avg,0.0
Bwd Bulk Rate Avg,0.0


The two backward flag fields and six bulk fields are zero throughout these data. Only those eight columns are excluded.

In [20]:
expected_constants = {
    "Bwd PSH Flags", "Bwd URG Flags", "Fwd Bytes/Bulk Avg", "Fwd Packet/Bulk Avg",
    "Fwd Bulk Rate Avg", "Bwd Bytes/Bulk Avg", "Bwd Packet/Bulk Avg", "Bwd Bulk Rate Avg",
}
if set(constant_values) != expected_constants:
    raise ValueError("The observed constant features differ from the documented eight; review before removal.")
columns_before = df_valid.shape[1]
df_valid.drop(columns=list(constant_values), inplace=True)
print(f"Columns retained: {df_valid.shape[1]} of {columns_before}.")

Columns retained: 76 of 84.


## 9. Recorded transport headers

In this section, we will compare the header bytes recorded in each direction with the number of packets. We check TCP and UDP separately because their headers have different lengths.

### 9.1 TCP header validation

A TCP header is 20–60 bytes long, and its length is a multiple of four ([RFC 9293](https://www.rfc-editor.org/info/rfc9293/)). We check the forward and backward totals against their respective packet counts. For example, for forward headers of 20, 24 and 32 bytes, `Fwd Header Length` is the sum (76 bytes), while `Fwd Seg Size Min` is the smallest individual header (20 bytes). With no forward packets there is no minimum to validate. The CSV provides no corresponding backward minimum.

The first table counts flows separately for each rule. A flow can fail more than one rule, so adding those counts would count some flows twice. The second table lists each flagged flow once.

In [21]:
def tcp_header_violations(frame):
    tcp = frame["Protocol"].eq(6)
    checks = {}
    for side, packets in (("Fwd", "Total Fwd Packet"), ("Bwd", "Total Bwd packets")):
        n, total = frame[packets], frame[f"{side} Header Length"]
        checks[f"{side} total below 20 bytes/packet"] = tcp & total.lt(20 * n)
        checks[f"{side} total above 60 bytes/packet"] = tcp & total.gt(60 * n)
        checks[f"{side} total not divisible by four"] = tcp & total.mod(4).ne(0)
    minimum = frame["Fwd Seg Size Min"]
    has_forward = tcp & frame["Total Fwd Packet"].gt(0)
    checks["Forward minimum outside 20..60"] = has_forward & ~minimum.between(20, 60)
    checks["Forward minimum not divisible by four"] = has_forward & minimum.mod(4).ne(0)
    return pd.DataFrame(checks, index=frame.index)


header_checks = tcp_header_violations(df_valid)
invalid_tcp_header_mask = header_checks.any(axis=1)
show_table(header_checks.sum().to_frame("Rows flagged (checks overlap)"), "Header check")
print(f"Distinct TCP rows flagged: {invalid_tcp_header_mask.sum():,}")
show_table(df_valid.loc[invalid_tcp_header_mask, [
    "Label", "Total Fwd Packet", "Fwd Header Length", "Fwd Seg Size Min",
    "Total Bwd packets", "Bwd Header Length",
]])

Header check,Rows flagged (checks overlap)
Fwd total below 20 bytes/packet,0
Fwd total above 60 bytes/packet,5
Fwd total not divisible by four,6
Bwd total below 20 bytes/packet,0
Bwd total above 60 bytes/packet,5
Bwd total not divisible by four,5
Forward minimum outside 20..60,1
Forward minimum not divisible by four,6


Distinct TCP rows flagged: 7


Label,Total Fwd Packet,Fwd Header Length,Fwd Seg Size Min,Total Bwd packets,Bwd Header Length
BENIGN,13.0,1723.0,38.0,20.0,5395.0
BENIGN,5.0,367.0,46.0,7.0,384.0
BENIGN,4.0,734.0,46.0,9.0,6602.0
BENIGN,4.0,169.0,31.0,2.0,92.0
BENIGN,3.0,949.0,126.0,6.0,5737.0
BENIGN,4.0,885.0,31.0,14.0,14521.0
BENIGN,6.0,132.0,20.0,1.0,1334.0


Six of the seven flows exceed the size limit in at least one direction. In the first row, 13 forward packets can account for at most 13 × 60 = 780 header bytes, yet the recorded total is 1,723.

The row with four forward packets shows why the size limit alone is not enough. Its 169 header bytes fall within the allowed range of 80–240. But each header length is divisible by four, so their sum must be as well. Neither the 169-byte total nor the 31-byte minimum is divisible by four.

### 9.2 Invalid TCP-header removal

These rows are excluded because their recorded measurements are inconsistent with TCP headers, regardless of the attack label.

In [22]:
remove_and_report(df_valid, invalid_tcp_header_mask)
del header_checks, invalid_tcp_header_mask

Label,Rows before removal,Rows removed,% of label removed,Rows remaining
BENIGN,2268589,7,0.0003,2268582
Total,2824951,7,0.0002,2824944


All seven excluded records were labeled BENIGN. Their removal was based on the recorded TCP header measurements, not the label.

### 9.3 UDP header measurements

Unlike TCP, UDP has a fixed eight-byte header ([RFC 768](https://www.rfc-editor.org/rfc/rfc768.html)). Four UDP packets therefore have exactly 32 bytes of UDP headers, not a range of possible totals. For each direction, we multiply the UDP packet count by eight and compare the result with the recorded header total.

In [23]:
udp = df_valid.loc[df_valid["Protocol"].eq(17), [
    "Total Fwd Packet", "Total Bwd packets", "Fwd Header Length", "Bwd Header Length"
]]
mismatch = pd.DataFrame({
    side: udp[f"{side} Header Length"].ne(8 * udp[packets])
    for side, packets in (("Fwd", "Total Fwd Packet"), ("Bwd", "Total Bwd packets"))
})
mismatch["Either direction"] = mismatch.any(axis=1)
udp_summary = mismatch.sum().rename("Flows not matching 8 bytes/packet").to_frame()
udp_summary["UDP flows checked"] = len(udp)
udp_summary["% of UDP flows"] = 100 * udp_summary.iloc[:, 0] / len(udp) if len(udp) else np.nan
show_table(udp_summary.round(4), "Direction")

Direction,Flows not matching 8 bytes/packet,UDP flows checked,% of UDP flows
Fwd,999416,999425,99.9991
Bwd,987910,999425,98.8478
Either direction,999417,999425,99.9992


For 999,417 of the 999,425 UDP flows, at least one direction fails the eight-bytes-per-packet comparison. With a mismatch this widespread, we cannot interpret these exported fields as literal UDP header totals. We keep the records rather than treating the mismatch as evidence against individual flows.

## 10. Packet counts, rates, and timing consistency

Measurements that describe the same flow should agree. For example, 10 packets over one second should give approximately 10 packets/s. Counts and structural limits are checked directly; derived rates, timings, and averages can differ because of how precisely the CSV stores them.

### 10.1 Packet counts and numeric ranges

This check will compare counts and codes against their allowed ranges and test relationships between fields. For example, if a flow contains five forward packets, at most five can carry a particular TCP flag. A discrepancy in a quantity derived by CICFlowMeter, such as its down/up ratio, calls for a closer look at how that quantity was calculated.

In [24]:
def exact_count_checks(x):
    f, b = x["Total Fwd Packet"], x["Total Bwd packets"]
    flags = [c for c in x if "Flag" in c]
    checks = {
        "Port or protocol outside encoded range": (
            ~x["Src Port"].between(0, 65535) | ~x["Dst Port"].between(0, 65535)
            | ~x["Protocol"].between(0, 255)),
        "No packets in the flow": (f + b).le(0),
        "Flag count exceeds applicable packet count": np.logical_or.reduce([
            x[c].gt(f if c.startswith("Fwd") else b if c.startswith("Bwd") else f + b)
            for c in flags]),
        "Non-TCP row has a positive TCP flag": x["Protocol"].ne(6) & x[flags].gt(0).any(axis=1),
        "Active forward data packets exceed packets or payload bytes":
            (x["Fwd Act Data Pkts"] > f) | (x["Fwd Act Data Pkts"] > x["Total Length of Fwd Packet"]),
        "Forward minimum header exceeds average": x["Fwd Seg Size Min"] * f > x["Fwd Header Length"],
        "Down/up ratio differs from integer packet ratio":
            x["Down/Up Ratio"].ne(b.floordiv(f.clip(lower=1)).where(f.gt(0), 0)),
        "Flow payload maximum differs from directional maximum":
            x["Packet Length Max"].ne(np.maximum(x["Fwd Packet Length Max"], x["Bwd Packet Length Max"])),
        "Flow payload minimum differs from minimum of populated directions":
            x["Packet Length Min"].ne(np.minimum(x["Fwd Packet Length Min"].where(f.gt(0), np.inf),
                                                x["Bwd Packet Length Min"].where(b.gt(0), np.inf))),
    }
    for side, n in (("Fwd", f), ("Bwd", b)):
        total = x[f"Total Length of {side} Packet"]
        low, high = x[f"{side} Packet Length Min"], x[f"{side} Packet Length Max"]
        checks[f"{side} payload total outside packet-count bounds"] = ~total.between(n * low, n * high)
        fields = [total.name, high.name, f"{side} Header Length", f"{side} Packets/s"]
        checks[f"{side} absent but has volume, header or rate"] = n.eq(0) & x[fields].ne(0).any(axis=1)
        checks[f"{side} has IAT but fewer than two packets"] = (
            n.le(1) & x.filter(regex=f"^{side} IAT ").ne(0).any(axis=1))
    return pd.Series({name: int(np.sum(mask)) for name, mask in checks.items()})


# Scan in chunks; do not keep a full-dataset boolean matrix for every diagnostic.
def audit_chunks():
    for start in range(0, len(df_valid), 100_000):
        yield df_valid.iloc[start:start + 100_000]


def review_exact_counts(counts):
    review_only = {
        "Forward minimum header exceeds average",
        "Down/up ratio differs from integer packet ratio",
        "Flow payload maximum differs from directional maximum",
        "Flow payload minimum differs from minimum of populated directions",
        "Fwd payload total outside packet-count bounds",
        "Bwd payload total outside packet-count bounds",
    }
    table = counts.rename("Flagged rows").to_frame()
    table["Interpretation if violated"] = np.where(table.index.isin(review_only),
                                                   "Review extractor calculation", "Invalid measurement")
    show_table(table, "Check")
    violations = counts[counts.gt(0) & ~counts.index.isin(review_only)]
    if not violations.empty:
        raise ValueError("Invalid exported measurements (affected rows per rule):\n" + violations.to_string())


review_exact_counts(sum(exact_count_checks(chunk) for chunk in audit_chunks()))

Check,Flagged rows,Interpretation if violated
Port or protocol outside encoded range,0,Invalid measurement
No packets in the flow,0,Invalid measurement
Flag count exceeds applicable packet count,0,Invalid measurement
Non-TCP row has a positive TCP flag,0,Invalid measurement
Active forward data packets exceed packets or payload bytes,0,Invalid measurement
Forward minimum header exceeds average,0,Review extractor calculation
Down/up ratio differs from integer packet ratio,0,Review extractor calculation
Flow payload maximum differs from directional maximum,0,Review extractor calculation
Flow payload minimum differs from minimum of populated directions,0,Review extractor calculation
Fwd payload total outside packet-count bounds,0,Review extractor calculation


No remaining flow violated a count or range rule, and none showed a discrepancy in the exact comparisons.

### 10.2 Rates, averages, and rounded measurements

We compare measurements that describe the same flow, allowing for the fact that a rounded rate or average may not exactly reproduce the total from which it was calculated. **Literal discrepancies** counts flows that fail a comparison before any allowance for CSV rounding. **Beyond rounding margin** counts those that still fail after allowing half a unit in the third significant digit of rounded inputs. Both comparisons use a numerical tolerance of `1e-6`; a recorded zero has no additional rounding allowance.

For example, four packets spanning 16 µs have three inter-arrival times with a mean of 16/3 µs. If the CSV records that mean as 5.33 µs, multiplying by three gives 15.99 µs instead of 16 µs. At this precision the mean can differ by up to 0.005 µs from rounding alone, or 0.015 µs across three intervals. The 0.01 µs discrepancy would therefore count in the first column but not the second. This tests whether rounding could explain a difference; it does not establish how every CSV field was written.

In [25]:
def rounding_margin(a):
    magnitude = np.abs(a)
    safe_magnitude = np.where(magnitude == 0, 1.0, magnitude)
    return np.where(magnitude == 0, 0.0, 0.5 * 10.0 ** (np.floor(np.log10(safe_magnitude)) - 2))


def rounded_checks(x):
    counts = {}

    def compare(name, difference, margin, *, equality=False):
        error = np.abs(difference) if equality else difference
        counts[name] = (int((error > 1e-6).sum()), int((error > margin + 1e-6).sum()))

    duration = x["Flow Duration"]
    f, b = x["Total Fwd Packet"], x["Total Bwd packets"]
    payload = x["Total Length of Fwd Packet"] + x["Total Length of Bwd Packet"]
    for side, n in (("Fwd", f), ("Bwd", b), ("Flow", f + b)):
        intervals = (n - 1).clip(lower=0)
        mean = x[f"{side} IAT Mean"]
        span = duration if side == "Flow" else x[f"{side} IAT Total"]
        compare(f"{side} IAT mean × intervals vs span", mean * intervals - span,
                rounding_margin(mean) * intervals + rounding_margin(span), equality=True)
        maximum = x[f"{side} IAT Max"]
        compare(f"{side} maximum IAT > duration", maximum - duration, rounding_margin(maximum))
        rate = x[f"{side} Packets/s"]
        compare(f"{side} packet rate vs count/duration", rate - n * 1e6 / duration,
                rounding_margin(rate), equality=True)
        if side != "Flow":
            total = x[f"Total Length of {side} Packet"]
            avg = x[f"{side} Packet Length Mean"]
            seg = x[f"{side} Segment Size Avg"]
            compare(f"{side} IAT span > duration", span - duration, rounding_margin(span))
            compare(f"{side} mean payload × packets vs total", avg * n - total,
                    rounding_margin(avg) * n + rounding_margin(total), equality=True)
            compare(f"{side} segment average vs packet mean", seg - avg,
                    rounding_margin(seg) + rounding_margin(avg), equality=True)
            compare(f"{side} subflow bytes > total", x[f"Subflow {side} Bytes"] - total,
                    rounding_margin(total))
            compare(f"{side} subflow packets > total", x[f"Subflow {side} Packets"] - n, 0)
    rate = x["Flow Bytes/s"]
    byte_margin = sum(rounding_margin(x[f"Total Length of {s} Packet"]) for s in ("Fwd", "Bwd"))
    compare("Flow byte rate vs bytes/duration", rate - payload * 1e6 / duration,
            rounding_margin(rate) + byte_margin * 1e6 / duration, equality=True)
    for stem in ("Flow IAT", "Fwd IAT", "Bwd IAT", "Fwd Packet Length", "Bwd Packet Length",
                 "Packet Length", "Active", "Idle"):
        lo, hi, avg, sd = (x[f"{stem} {suffix}"] for suffix in ("Min", "Max", "Mean", "Std"))
        # For sample standard deviation with at least two observations, s <= range / sqrt(2).
        compare(f"{stem}: mean below min", lo - avg, rounding_margin(lo) + rounding_margin(avg))
        compare(f"{stem}: mean above max", avg - hi, rounding_margin(avg) + rounding_margin(hi))
        compare(f"{stem}: std above range bound", sd - (hi - lo) / np.sqrt(2),
                rounding_margin(sd) + (rounding_margin(hi) + rounding_margin(lo)) / np.sqrt(2))
        if stem in ("Idle", "Active"):
            compare(f"{stem}: maximum > duration", hi - duration, rounding_margin(hi))
    sd, var = x["Packet Length Std"], x["Packet Length Variance"]
    compare("Packet-length variance vs squared std", var - sd**2,
            rounding_margin(var) + 2 * sd * rounding_margin(sd) + rounding_margin(sd)**2,
            equality=True)
    return pd.DataFrame.from_dict(counts, orient="index", columns=["Literal discrepancies", "Beyond rounding margin"])


derived_audit = sum(rounded_checks(chunk) for chunk in audit_chunks())
print(f"Checked {len(derived_audit)} relationships; displaying those with discrepancies.")
show_table(derived_audit.loc[derived_audit["Literal discrepancies"].gt(0)], "Relationship")

Checked 47 relationships; displaying those with discrepancies.


Relationship,Literal discrepancies,Beyond rounding margin
Fwd IAT mean × intervals vs span,494302,0
Fwd maximum IAT > duration,25580,0
Fwd packet rate vs count/duration,509629,0
Fwd IAT span > duration,120417,0
Fwd mean payload × packets vs total,15632,0
Bwd IAT mean × intervals vs span,366112,0
Bwd maximum IAT > duration,12667,0
Bwd packet rate vs count/duration,382174,0
Bwd IAT span > duration,55988,0
Bwd mean payload × packets vs total,100460,0


For the whole-flow packet rate, 628,936 recorded values differ from the rate calculated as packet count divided by duration in seconds. None of those differences exceeds the rounding allowance. The same distinction holds across all 47 relationships: no discrepancy remains beyond the tested margins, so these checks do not justify removing any flows.

## 11. Duplicate records and label conflicts

Matching packet counts alone do not make two records duplicates. We compare every recorded input, including the endpoints, timestamp and flow measurements. If three rows match on all inputs and have the same label, we keep one and remove two copies. If their labels differ, we flag a conflict rather than choosing a label.

### 11.1 Repeated inputs and their labels

In [26]:
def duplicate_group_counts(frame):
    recorded = frame.columns.drop("Label").tolist()
    repeated = frame.loc[frame.duplicated(subset=recorded, keep=False)]
    # Only repeated recorded inputs need a label comparison.
    groups = repeated.groupby(recorded, sort=False, dropna=False, observed=True)["Label"]
    sizes = groups.size()
    conflicting = groups.nunique(dropna=False).gt(1)
    same_label_sizes = sizes[~conflicting]
    return pd.Series({
        "Rows in same-label duplicate groups": int(same_label_sizes.sum()),
        "Distinct same-label duplicate groups": len(same_label_sizes),
        "Removable extra copies": int((same_label_sizes - 1).sum()),
        "Groups with conflicting labels": int(conflicting.sum()),
        "Rows belonging to conflicting-label groups": int(sizes[conflicting].sum()),
    }, name="Count")


duplicate_audit = duplicate_group_counts(df_valid)
show_table(duplicate_audit.to_frame(), "Duplicate check")
removable_extra_copy_count = duplicate_audit["Removable extra copies"]
if duplicate_audit["Groups with conflicting labels"]:
    raise ValueError("Identical recorded inputs have different labels; investigate before removal.")

Duplicate check,Count
Rows in same-label duplicate groups,301
Distinct same-label duplicate groups,102
Removable extra copies,199
Groups with conflicting labels,0
Rows belonging to conflicting-label groups,0


The 102 repeated groups contain 301 records, and the labels agree within every group. Retaining one record per group leaves 301 − 102 = 199 extra copies.

### 11.2 Exact duplicate removal

Only extra copies of complete, same-label records are removed. The number identified for removal must match the duplicate audit.

In [27]:
exact_duplicate_removal_mask = df_valid.duplicated(keep="first")
if int(exact_duplicate_removal_mask.sum()) != removable_extra_copy_count:
    raise ValueError("The exact-duplicate count does not match the audit.")
remove_and_report(df_valid, exact_duplicate_removal_mask)

Label,Rows before removal,Rows removed,% of label removed,Rows remaining
BENIGN,2268582,198,0.0087,2268384
DoS Hulk,229965,1,0.0004,229964
Total,2824944,199,0.0070,2824745


Keeping the first copy removed 198 BENIGN records and one DoS Hulk record. No conflicting label had to be resolved.

## 12. Final validation and export

Before saving the cleaned data, we check which columns and labels remain and compare the final class counts with the original labeled population.

### 12.1 Integrity and class support

The checks compare the cleaned table with the expected 71 numeric features, four metadata columns and 15 labels. The class table compares the final counts with the original labeled population, so the cumulative effect of all removal steps is visible.

In [28]:
def validate_export_schema(frame):
    text_columns = {"Flow ID", "Src IP", "Dst IP", "Timestamp", "Label"}
    stats = ("Min", "Max", "Mean", "Std")
    numeric_columns = {
        "Src Port", "Dst Port", "Protocol", "Flow Duration", "Total Fwd Packet", "Total Bwd packets",
        "Flow Bytes/s", "Flow Packets/s", "Fwd PSH Flags", "Fwd URG Flags", "Down/Up Ratio",
        "Average Packet Size", "Packet Length Variance", "FWD Init Win Bytes", "Bwd Init Win Bytes",
        "Fwd Act Data Pkts", "Fwd Seg Size Min",
        *(f"{stem} {stat}" for stem in ("Flow IAT", "Fwd IAT", "Bwd IAT", "Fwd Packet Length",
                                        "Bwd Packet Length", "Packet Length", "Active", "Idle") for stat in stats),
        *(f"{flag} Flag Count" for flag in ("FIN", "SYN", "RST", "PSH", "ACK", "URG", "CWR", "ECE")),
        *(pattern.format(side) for side in ("Fwd", "Bwd") for pattern in (
            "Total Length of {} Packet", "{} IAT Total", "{} Header Length", "{} Packets/s",
            "{} Segment Size Avg", "Subflow {} Packets", "Subflow {} Bytes")),
    }
    expected_labels = {
        "BENIGN", "Bot", "DDoS", "DoS GoldenEye", "DoS Hulk", "DoS Slowhttptest", "DoS slowloris",
        "FTP-Patator", "Heartbleed", "Infiltration", "PortScan", "SSH-Patator",
        "Web Attack - Brute Force", "Web Attack - Sql Injection", "Web Attack - XSS",
    }
    expected = numeric_columns | text_columns
    actual = set(frame.columns)
    repeated = frame.columns[frame.columns.duplicated()].tolist()
    if expected != actual or repeated:
        raise ValueError(f"Export columns: missing={sorted(expected - actual)}, "
                         f"unexpected={sorted(actual - expected)}, duplicate={repeated}")

    wrong_numeric = sorted(c for c in numeric_columns if not (
        pd.api.types.is_integer_dtype(frame[c]) or pd.api.types.is_float_dtype(frame[c])))
    wrong_text = sorted(c for c in text_columns
                        if pd.api.types.infer_dtype(frame[c], skipna=True) not in ('string', 'empty'))
    if wrong_numeric or wrong_text:
        raise ValueError(f"Unexpected column types: expected numeric={wrong_numeric}, expected text={wrong_text}")

    labels = frame["Label"]
    observed = set(labels.dropna().unique())
    if observed != expected_labels or labels.hasnans:
        raise ValueError(f"Export labels: missing={sorted(expected_labels - observed)}, "
                         f"unexpected={sorted(observed - expected_labels)}, nulls={labels.isna().sum()}")


validate_export_schema(df_valid)
final_numeric_features = df_valid.select_dtypes(include="number").columns.tolist()
final_window_features = ["FWD Init Win Bytes", "Bwd Init Win Bytes"]
final_nonnegative_features = [
    feature
    for feature in final_numeric_features
    if feature not in final_window_features
]

final_missing_cells = int(df_valid.isna().sum().sum())
final_infinite_cells = int(
    np.isinf(df_valid[final_numeric_features]).sum().sum()
)
final_nonpositive_duration_rows = int(df_valid["Flow Duration"].le(0).sum())
final_unexpected_negative_cells = int(
    df_valid[final_nonnegative_features].lt(0).sum().sum()
)
final_invalid_window_cells = int(
    (
        df_valid[final_window_features].lt(-1)
        | df_valid[final_window_features].gt(65535)
    )
    .sum()
    .sum()
)
final_exact_duplicate_rows = int(df_valid.duplicated(keep=False).sum())
final_conflicting_feature_rows = duplicate_group_counts(df_valid)["Rows belonging to conflicting-label groups"]

final_dataset_dimensions = pd.DataFrame(
    {"Count": [len(df_valid), df_valid.shape[1]]},
    index=["Rows after all cleaning", "Columns after all cleaning"],
)
final_integrity_checks = pd.DataFrame(
    {
        "Observed count": [
            int(df_valid["Label"].isna().sum()),
            final_missing_cells,
            final_infinite_cells,
            final_nonpositive_duration_rows,
            final_unexpected_negative_cells,
            final_invalid_window_cells,
            int(tcp_header_violations(df_valid).any(axis=1).sum()),
            final_exact_duplicate_rows,
            final_conflicting_feature_rows,
        ],
        "Expected count": 0,
    },
    index=[
        "Missing labels",
        "Missing cells (entire dataset)",
        "Infinite numeric cells",
        "Rows with zero or negative duration",
        "Unexpected negative numeric cells",
        "Invalid TCP-window cells",
        "Rows with invalid recorded TCP headers",
        "Rows belonging to exact-duplicate groups",
        "Rows belonging to conflicting-label groups",
    ],
)
show_table(final_dataset_dimensions, "Dimension")
show_table(final_integrity_checks, "Integrity check")

validation_checks = final_integrity_checks["Observed count"]
if validation_checks.ne(0).any():
    raise AssertionError(
        "Final validation failed:\n"
        + validation_checks[validation_checks.ne(0)].to_string()
    )

final_counts = df_valid["Label"].value_counts().reindex(label_counts_before_cleaning.index, fill_value=0)
final_label_distribution = pd.DataFrame({
    "Originally labeled rows": label_counts_before_cleaning,
    "Rows removed": label_counts_before_cleaning - final_counts,
    "% of label removed": 100 * (label_counts_before_cleaning - final_counts) / label_counts_before_cleaning,
    "Final rows": final_counts,
    "% of final dataset": 100 * final_counts / len(df_valid),
})
show_table(final_label_distribution.round(4), "Label")
print("All final data-integrity checks passed.")

Dimension,Count
Rows after all cleaning,2824745
Columns after all cleaning,76


Integrity check,Observed count,Expected count
Missing labels,0,0
Missing cells (entire dataset),0,0
Infinite numeric cells,0,0
Rows with zero or negative duration,0,0
Unexpected negative numeric cells,0,0
Invalid TCP-window cells,0,0
Rows with invalid recorded TCP headers,0,0
Rows belonging to exact-duplicate groups,0,0
Rows belonging to conflicting-label groups,0,0


Label,Originally labeled rows,Rows removed,% of label removed,Final rows,% of final dataset
BENIGN,2273097,4713,0.2073,2268384,80.3040
DoS Hulk,231073,1109,0.4799,229964,8.1411
PortScan,158930,126,0.0793,158804,5.6219
DDoS,128027,21,0.0164,128006,4.5316
DoS GoldenEye,10293,5,0.0486,10288,0.3642
FTP-Patator,7938,7,0.0882,7931,0.2808
SSH-Patator,5897,2,0.0339,5895,0.2087
DoS slowloris,5796,0,0.0000,5796,0.2052
DoS Slowhttptest,5499,0,0.0000,5499,0.1947
Bot,1966,10,0.5086,1956,0.0692


All final data-integrity checks passed.


Every listed final integrity counter is zero: there are no missing labels or feature values, infinities, nonpositive durations, unexpected negatives, invalid TCP-window values, exact duplicates or complete-feature label conflicts. This confirms the listed rules, not the absence of every possible extractor error. Class imbalance remains substantial: `BENIGN` still represents `80.30%` of flows, while Heartbleed has only 7 rows, SQL Injection 21 and Infiltration 35.

### 12.2 Cleaned dataset export

In [29]:
PROCESSED_DATA_PATH = (
    DATASET_PATH.parent.parent
    / "processed"
    / "cicids2017_cleaned.parquet"
)
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_valid.to_parquet(PROCESSED_DATA_PATH, index=False)

if not PROCESSED_DATA_PATH.is_file():
    raise FileNotFoundError("The cleaned Parquet file was not created.")

print(f"Saved cleaned dataset: {PROCESSED_DATA_PATH.relative_to(DATASET_PATH.parents[3]).as_posix()}")
print(f"Rows saved: {len(df_valid):,}")
print(f"Columns saved: {df_valid.shape[1]}")
print(
    "File size: "
    f"{PROCESSED_DATA_PATH.stat().st_size / (1024 ** 2):,.2f} MiB"
)

Saved cleaned dataset: ml/data/processed/cicids2017_cleaned.parquet
Rows saved: 2,824,745
Columns saved: 76
File size: 355.91 MiB


## Final EDA summary

**Starting population.** Of the 3,119,345 rows in the merged CSV, 288,602 had no `Label` and were excluded. This left 2,830,743 labeled flows across 15 classes. We corrected the separator in three Web Attack labels without changing the attack names. We also checked the 44 fields expected to contain whole numbers: none had a fractional finite value, so none needed rounding.

**What we removed.** Missing values and infinities occurred only in the two rate columns, on the same 2,867 zero-duration flows. Most had packet timings consistent with a shared timestamp. We excluded them because we could not calculate their rates, not because a zero duration proves a flow is corrupt. Among the remaining flows, 2,925 had negative timing, header or segment measurements, and seven had header measurements inconsistent with TCP. We also removed eight columns that were always zero and 199 extra copies of identical records. None of the repeated inputs had conflicting labels.

**What we kept.** A TCP window of −1 means the value is unavailable. For UDP, the recorded header totals generally did not behave like eight bytes per packet. Neither observation justified deleting flows. We also kept records where derived rates, timings or averages differed only within the rounding margins we tested.

**Final dataset.** The cleaned table contains **2,824,745 flows, 76 columns and all 15 labels**, with no missing cells or infinite numeric values. BENIGN still accounts for 80.30% of flows. For Heartbleed, the removal of four records reduced the class from eleven flows to seven.